# Red Piezométrica de Catalunya — Análisis de Aguas Subterráneas

**Autor:** Carlos Daniel Muñoz Sánchez  
**Fuente de datos:** Agència Catalana de l'Aigua (ACA) — Datos Abiertos  
**Dataset:** Nivell piezomètric de les aigües subterrànies de Catalunya  
**URL:** https://analisi.transparenciacatalunya.cat/api/views/6899-hrme/rows.csv  
**Licencia:** Datos abiertos Generalitat de Catalunya  

---

## Objetivos

1. **Descargar** datos piezométricos históricos de la red de monitoreo del ACA directamente desde su API pública.
2. **Explorar y limpiar** el dataset — calidad de datos, cobertura temporal y espacial.
3. **Analizar tendencias temporales** — ¿están subiendo o bajando los niveles de agua subterránea en Catalunya?
4. **Visualizar espacialmente** — mapa interactivo con Folium mostrando la red de piezómetros y sus tendencias.
5. **Identificar patrones** — comparación por masa de agua y variabilidad estacional.

---

## Contexto hidrogeológico

Catalunya cuenta con una red de monitoreo piezométrico gestionada por la ACA que registra los niveles del agua subterránea en decenas de pozos distribuidos por las diferentes masas de agua subterránea de las cuencas internas catalanas. Este seguimiento es fundamental para:

- Detectar situaciones de sobreexplotación de acuíferos
- Evaluar el impacto de las sequías en los recursos hídricos subterráneos
- Planificar extracciones sostenibles
- Monitorear la recuperación de acuíferos tras períodos de restricción

El nivel piezométrico (expresado en metros sobre el nivel del mar) es el indicador principal del estado cuantitativo de un acuífero.

In [ ]:
# ── LIBRERÍAS ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from scipy.stats import linregress
import folium
from folium.plugins import MarkerCluster
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.labelsize': 11,
})

print('✓ Librerías cargadas correctamente')

---
## 1. Descarga de datos desde la API del ACA

In [ ]:
# ── DESCARGA DIRECTA DESDE LA API PÚBLICA DEL ACA ─────────────────────────────
# Dataset: Nivell piezomètric de les aigües subterrànies de Catalunya
# Fuente: Transparència Catalunya / ACA
# Licencia: Dades obertes Generalitat de Catalunya (CC BY 4.0)

URL_ACA = (
    'https://analisi.transparenciacatalunya.cat'
    '/api/views/6899-hrme/rows.csv?accessType=DOWNLOAD'
)

print('Descargando datos de la red piezométrica del ACA...')
print(f'URL: {URL_ACA}\n')

try:
    df_raw = pd.read_csv(URL_ACA)
    print(f'✓ Datos descargados correctamente')
    print(f'  → Filas:    {len(df_raw):,}')
    print(f'  → Columnas: {df_raw.shape[1]}')
    print(f'\nColumnas disponibles:')
    for col in df_raw.columns:
        print(f'  • {col}')
except Exception as e:
    print(f'Error en la descarga: {e}')
    print('Verifica tu conexión a internet e intenta de nuevo.')

In [ ]:
# Vista previa de los datos crudos
print('PRIMERAS FILAS DEL DATASET:')
df_raw.head(10)

---
## 2. Exploración y limpieza de datos

In [ ]:
# ── EXPLORACIÓN INICIAL ────────────────────────────────────────────────────────
print('INFORMACIÓN GENERAL DEL DATASET')
print('=' * 50)
print(f'Período:          {df_raw.iloc[:,0].min()} → {df_raw.iloc[:,0].max()}')
print(f'Total registros:  {len(df_raw):,}')
print(f'\nTipos de datos:')
print(df_raw.dtypes)
print(f'\nValores nulos por columna:')
print(df_raw.isnull().sum())

In [ ]:
# ── LIMPIEZA Y ESTANDARIZACIÓN ─────────────────────────────────────────────────
df = df_raw.copy()

# Estandarizar nombres de columnas
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Identificar columnas clave automáticamente
# (el dataset puede variar ligeramente en nombres)
col_map = {}
for col in df.columns:
    if 'data' in col or 'fecha' in col or 'date' in col:
        col_map['fecha'] = col
    elif 'estaci' in col or 'estacion' in col or 'station' in col:
        col_map['estacion'] = col
    elif 'massa' in col or 'masa' in col or 'mass' in col:
        col_map['masa_agua'] = col
    elif 'utm_x' in col or 'coord_x' in col or 'x_utm' in col:
        col_map['x'] = col
    elif 'utm_y' in col or 'coord_y' in col or 'y_utm' in col:
        col_map['y'] = col
    elif 'fondaria' in col or 'profundidad' in col or 'depth' in col:
        col_map['profundidad'] = col
    elif 'nivell' in col or 'nivel' in col or 'level' in col or 'alcada' in col or 'altura' in col:
        col_map['nivel'] = col

print('Mapeo de columnas identificadas:')
for k, v in col_map.items():
    print(f'  {k:15s} → {v}')

# Renombrar a nombres estándar
df = df.rename(columns={v: k for k, v in col_map.items()})

# Convertir fecha
df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')

# Convertir nivel a numérico
df['nivel'] = pd.to_numeric(df['nivel'], errors='coerce')

# Convertir coordenadas
df['x'] = pd.to_numeric(df['x'], errors='coerce')
df['y'] = pd.to_numeric(df['y'], errors='coerce')

# Eliminar filas sin datos esenciales
df = df.dropna(subset=['fecha', 'nivel', 'x', 'y'])

# Extraer año y mes
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

print(f'\n✓ Dataset limpio: {len(df):,} registros válidos')
print(f'  → Período:       {df["fecha"].min().date()} → {df["fecha"].max().date()}')
print(f'  → Estaciones:    {df["estacion"].nunique():,} piezómetros únicos')
print(f'  → Masas de agua: {df["masa_agua"].nunique()} masas de agua')
print(f'  → Nivel mín/máx: {df["nivel"].min():.1f} / {df["nivel"].max():.1f} m.s.n.m.')

---
## 3. Análisis exploratorio

In [ ]:
# ── DISTRIBUCIÓN TEMPORAL Y ESPACIAL ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Registros por año
registros_año = df.groupby('año').size()
axes[0].bar(registros_año.index, registros_año.values, color='#2a9d8f', alpha=0.85)
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Número de mediciones')
axes[0].set_title('Registros por año', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# 2. Distribución de niveles piezométricos
axes[1].hist(df['nivel'], bins=50, color='#457b9d', alpha=0.85, edgecolor='white')
axes[1].axvline(df['nivel'].median(), color='#e63946', lw=2, ls='--',
                label=f'Mediana: {df["nivel"].median():.1f} m')
axes[1].set_xlabel('Nivel piezométrico (m.s.n.m.)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de niveles', fontweight='bold')
axes[1].legend()

# 3. Mediciones por masa de agua (top 10)
top_masas = df['masa_agua'].value_counts().head(10)
axes[2].barh(range(len(top_masas)), top_masas.values, color='#e9c46a', alpha=0.85)
axes[2].set_yticks(range(len(top_masas)))
axes[2].set_yticklabels([m[:35] + '...' if len(m) > 35 else m
                         for m in top_masas.index], fontsize=8)
axes[2].set_xlabel('Número de registros')
axes[2].set_title('Top 10 masas de agua\n(por nº de mediciones)', fontweight='bold')

fig.suptitle('Análisis Exploratorio — Red Piezométrica ACA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/01_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: output/01_exploratorio.png')

---
## 4. Análisis de tendencias temporales

In [ ]:
# ── TENDENCIA GENERAL DE LA RED ────────────────────────────────────────────────
# Nivel promedio mensual de toda la red
tendencia_mensual = (
    df.groupby(df['fecha'].dt.to_period('M'))['nivel']
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
)
tendencia_mensual['fecha'] = tendencia_mensual['fecha'].dt.to_timestamp()
tendencia_mensual = tendencia_mensual[tendencia_mensual['count'] >= 5]  # mínimo 5 estaciones

# Ajuste lineal de tendencia general
x_num = (tendencia_mensual['fecha'] - tendencia_mensual['fecha'].min()).dt.days
slope, intercept, r_val, p_val, se = linregress(x_num, tendencia_mensual['mean'])
tendencia_lineal = slope * x_num + intercept
tendencia_anual = slope * 365  # m/año

fig, ax = plt.subplots(figsize=(14, 6))

# Banda de incertidumbre (±1 std)
ax.fill_between(
    tendencia_mensual['fecha'],
    tendencia_mensual['mean'] - tendencia_mensual['std'],
    tendencia_mensual['mean'] + tendencia_mensual['std'],
    alpha=0.2, color='#457b9d', label='±1 Desv. estándar'
)

# Serie temporal
ax.plot(tendencia_mensual['fecha'], tendencia_mensual['mean'],
        color='#457b9d', lw=1.5, alpha=0.8, label='Media mensual red')
ax.plot(tendencia_mensual['fecha'], tendencia_mensual['median'],
        color='#2a9d8f', lw=1.5, ls='--', alpha=0.8, label='Mediana mensual')

# Tendencia lineal
color_tend = '#e63946' if tendencia_anual < 0 else '#2a9d8f'
ax.plot(tendencia_mensual['fecha'], tendencia_lineal,
        color=color_tend, lw=2.5, ls='-',
        label=f'Tendencia: {tendencia_anual:+.3f} m/año (R²={r_val**2:.2f}, p={p_val:.3f})')

ax.set_xlabel('Fecha')
ax.set_ylabel('Nivel piezométrico promedio (m.s.n.m.)')
ax.set_title('Evolución temporal de los niveles piezométricos — Red ACA Catalunya\n'
             'Media mensual de todos los piezómetros activos', fontweight='bold')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.xticks(rotation=45)

# Anotación de tendencia
tendencia_txt = 'DESCENSO' if tendencia_anual < 0 else 'ASCENSO'
ax.text(0.02, 0.05, f'Tendencia general: {tendencia_txt}\n{tendencia_anual:+.3f} m/año',
        transform=ax.transAxes, fontsize=10, fontweight='bold',
        color=color_tend,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('output/02_tendencia_general.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: output/02_tendencia_general.png')
print(f'\nTendencia general de la red: {tendencia_anual:+.4f} m/año')
print(f'p-valor: {p_val:.4f} → {"estadísticamente significativa" if p_val < 0.05 else "no significativa"}')

In [ ]:
# ── TENDENCIA POR ESTACIÓN INDIVIDUAL ─────────────────────────────────────────
# Calcular tendencia de cada piezómetro con regresión lineal

def calcular_tendencia(grupo):
    """Calcula la tendencia lineal (m/año) para una estación."""
    grupo = grupo.sort_values('fecha').dropna(subset=['nivel'])
    if len(grupo) < 10:  # mínimo 10 mediciones para calcular tendencia
        return None
    x = (grupo['fecha'] - grupo['fecha'].min()).dt.days
    y = grupo['nivel']
    slope, intercept, r_val, p_val, _ = linregress(x, y)
    return pd.Series({
        'tendencia_m_año': slope * 365,
        'r2': r_val ** 2,
        'p_valor': p_val,
        'n_mediciones': len(grupo),
        'nivel_medio': grupo['nivel'].mean(),
        'nivel_min': grupo['nivel'].min(),
        'nivel_max': grupo['nivel'].max(),
        'rango_m': grupo['nivel'].max() - grupo['nivel'].min(),
        'año_inicio': grupo['fecha'].min().year,
        'año_fin': grupo['fecha'].max().year,
        'x': grupo['x'].iloc[0],
        'y': grupo['y'].iloc[0],
        'masa_agua': grupo['masa_agua'].iloc[0],
    })

print('Calculando tendencias por estación...')
tendencias = (
    df.groupby('estacion')
    .apply(calcular_tendencia)
    .dropna()
    .reset_index()
)

# Filtrar solo tendencias estadísticamente significativas (p < 0.10)
tend_sig = tendencias[tendencias['p_valor'] < 0.10].copy()

print(f'\n✓ Estaciones analizadas:           {len(tendencias)}')
print(f'  → Con tendencia significativa:   {len(tend_sig)} (p < 0.10)')
print(f'  → Con descenso significativo:    {(tend_sig["tendencia_m_año"] < 0).sum()}')
print(f'  → Con ascenso significativo:     {(tend_sig["tendencia_m_año"] > 0).sum()}')
print(f'\nTendencia media de la red: {tendencias["tendencia_m_año"].mean():+.3f} m/año')

In [ ]:
# ── DISTRIBUCIÓN DE TENDENCIAS ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de tendencias
colors_hist = ['#e63946' if t < 0 else '#2a9d8f' for t in tendencias['tendencia_m_año']]
n, bins, patches = axes[0].hist(tendencias['tendencia_m_año'], bins=40,
                                 edgecolor='white', alpha=0.85)
for patch, color in zip(patches,
    ['#e63946' if (bins[i] + bins[i+1])/2 < 0 else '#2a9d8f'
     for i in range(len(patches))]):
    patch.set_facecolor(color)

axes[0].axvline(0, color='black', lw=1.5, ls='--', label='Sin tendencia')
axes[0].axvline(tendencias['tendencia_m_año'].mean(), color='orange', lw=2,
                label=f'Media: {tendencias["tendencia_m_año"].mean():+.3f} m/año')
axes[0].set_xlabel('Tendencia (m/año)')
axes[0].set_ylabel('Número de estaciones')
axes[0].set_title('Distribución de tendencias\npor estación', fontweight='bold')
axes[0].legend()

# Tendencia por masa de agua
tend_masa = (
    tendencias.groupby('masa_agua')['tendencia_m_año']
    .agg(['mean', 'count'])
    .sort_values('mean')
)
tend_masa = tend_masa[tend_masa['count'] >= 2]  # mín 2 estaciones
colors_bar = ['#e63946' if v < 0 else '#2a9d8f' for v in tend_masa['mean']]
bars = axes[1].barh(range(len(tend_masa)), tend_masa['mean'],
                     color=colors_bar, alpha=0.85)
axes[1].set_yticks(range(len(tend_masa)))
axes[1].set_yticklabels(
    [f"{m[:30]}... (n={n})" if len(m) > 30 else f"{m} (n={n})"
     for m, n in zip(tend_masa.index, tend_masa['count'])],
    fontsize=8
)
axes[1].axvline(0, color='black', lw=1, ls='--')
axes[1].set_xlabel('Tendencia media (m/año)')
axes[1].set_title('Tendencia media por masa de agua\n(mín. 2 estaciones)', fontweight='bold')

fig.suptitle('Análisis de Tendencias — Piezómetros ACA Catalunya', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/03_distribucion_tendencias.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: output/03_distribucion_tendencias.png')

In [ ]:
# ── SERIES TEMPORALES DE LAS 6 ESTACIONES CON MÁS DATOS ───────────────────────
top_estaciones = (
    df.groupby('estacion').size()
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

palette = ['#e63946', '#2a9d8f', '#457b9d', '#e9c46a', '#f4a261', '#264653']

for i, estacion in enumerate(top_estaciones):
    datos_est = df[df['estacion'] == estacion].sort_values('fecha')
    masa = datos_est['masa_agua'].iloc[0] if 'masa_agua' in datos_est.columns else ''

    # Suavizado (media móvil 3 meses)
    datos_est = datos_est.set_index('fecha')
    suavizado = datos_est['nivel'].rolling(window=3, center=True).mean()

    # Tendencia
    x_num = (datos_est.index - datos_est.index.min()).days
    slope, intercept, r_val, p_val, _ = linregress(x_num, datos_est['nivel'])
    tend_anual = slope * 365
    tend_linea = slope * x_num + intercept

    color = palette[i]
    axes[i].plot(datos_est.index, datos_est['nivel'],
                 'o', ms=2, alpha=0.4, color=color)
    axes[i].plot(datos_est.index, suavizado,
                 '-', lw=2, color=color, label='Media móvil 3M')
    axes[i].plot(datos_est.index, tend_linea, '--k', lw=1.5,
                 label=f'Tendencia: {tend_anual:+.3f} m/año')

    axes[i].set_title(f'{estacion}\n{masa[:40] if masa else ""}',
                      fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Nivel (m.s.n.m.)')
    axes[i].legend(fontsize=7)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[i].xaxis.set_major_locator(mdates.YearLocator(3))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45)

fig.suptitle('Series Temporales — Estaciones con Mayor Cobertura Histórica\nRed Piezométrica ACA',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/04_series_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: output/04_series_temporales.png')

In [ ]:
# ── VARIABILIDAD ESTACIONAL ────────────────────────────────────────────────────
# Anomalía mensual respecto a la media anual (para eliminar tendencia)
df_estacional = df.copy()
media_estacion = df_estacional.groupby('estacion')['nivel'].transform('mean')
df_estacional['anomalia'] = df_estacional['nivel'] - media_estacion

estacional = df_estacional.groupby('mes')['anomalia'].agg(['mean', 'std'])

meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
         'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, ax = plt.subplots(figsize=(12, 5))

colors_mes = ['#2a9d8f' if v >= 0 else '#e63946' for v in estacional['mean']]
bars = ax.bar(range(1, 13), estacional['mean'], color=colors_mes, alpha=0.85,
              yerr=estacional['std'] / 2, capsize=4, error_kw={'alpha': 0.5})
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses)
ax.set_ylabel('Anomalía piezométrica media (m)')
ax.set_title('Variabilidad Estacional — Ciclo Anual de los Niveles Piezométricos\n'
             'Anomalía respecto a la media de cada estación (±½ desv. estándar)',
             fontweight='bold')

# Anotación interpretativa
mes_max = estacional['mean'].idxmax()
mes_min = estacional['mean'].idxmin()
ax.annotate(f'Máximo\n{meses[mes_max-1]}',
            xy=(mes_max, estacional['mean'].max()),
            xytext=(mes_max, estacional['mean'].max() + 0.3),
            fontsize=9, ha='center', color='#2a9d8f',
            arrowprops=dict(arrowstyle='->', color='#2a9d8f'))
ax.annotate(f'Mínimo\n{meses[mes_min-1]}',
            xy=(mes_min, estacional['mean'].min()),
            xytext=(mes_min, estacional['mean'].min() - 0.3),
            fontsize=9, ha='center', color='#e63946',
            arrowprops=dict(arrowstyle='->', color='#e63946'))

plt.tight_layout()
plt.savefig('output/05_estacionalidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: output/05_estacionalidad.png')

---
## 5. Mapa interactivo con Folium

In [ ]:
# ── CONVERSIÓN DE COORDENADAS UTM → WGS84 ─────────────────────────────────────
# Las coordenadas del ACA están en UTM zona 31N (EPSG:25831)
# Necesitamos convertir a lat/lon para Folium

try:
    import pyproj
    transformer = pyproj.Transformer.from_crs('EPSG:25831', 'EPSG:4326', always_xy=True)
    lon, lat = transformer.transform(
        tendencias['x'].values,
        tendencias['y'].values
    )
    tendencias['lat'] = lat
    tendencias['lon'] = lon
    print('✓ Coordenadas convertidas con pyproj (UTM 31N → WGS84)')
except ImportError:
    # Conversión aproximada manual para UTM 31N
    # Factor aproximado válido para Catalunya
    tendencias['lat'] = (tendencias['y'] - 4500000) / 111320 + 40.5
    tendencias['lon'] = (tendencias['x'] - 350000) / (111320 * 0.766) + 2.0
    print('⚠ pyproj no disponible — usando conversión aproximada')
    print('  Para resultados exactos: pip install pyproj')

# Filtrar coordenadas válidas (dentro de Catalunya aproximadamente)
mask = (
    (tendencias['lat'] > 40.0) & (tendencias['lat'] < 43.0) &
    (tendencias['lon'] > 0.0)  & (tendencias['lon'] < 4.0)
)
tendencias_mapa = tendencias[mask].copy()
print(f'✓ {len(tendencias_mapa)} estaciones con coordenadas válidas para el mapa')

In [ ]:
# ── MAPA INTERACTIVO ───────────────────────────────────────────────────────────
# Centro aproximado de Catalunya
mapa = folium.Map(
    location=[41.8, 1.8],
    zoom_start=8,
    tiles='CartoDB positron'
)

# Escala de colores basada en tendencia
# Rojo = descenso fuerte, Amarillo = estable, Verde = ascenso
def tendencia_color(tend):
    if tend < -0.5:   return '#d62728'   # descenso fuerte
    elif tend < -0.1: return '#ff7f0e'   # descenso moderado
    elif tend < 0.1:  return '#bcbd22'   # estable
    elif tend < 0.5:  return '#2ca02c'   # ascenso moderado
    else:             return '#1f77b4'   # ascenso fuerte

def tendencia_texto(tend):
    if tend < -0.5:   return '↓↓ Descenso fuerte'
    elif tend < -0.1: return '↓ Descenso moderado'
    elif tend < 0.1:  return '→ Estable'
    elif tend < 0.5:  return '↑ Ascenso moderado'
    else:             return '↑↑ Ascenso fuerte'

# Añadir cluster de marcadores
cluster = MarkerCluster(name='Piezómetros').add_to(mapa)

for _, row in tendencias_mapa.iterrows():
    color = tendencia_color(row['tendencia_m_año'])
    texto_tend = tendencia_texto(row['tendencia_m_año'])

    popup_html = f"""
    <div style='font-family: Arial; width: 220px;'>
        <h4 style='margin:0; color:#1d3557;'>{row['estacion']}</h4>
        <hr style='margin:4px 0;'>
        <b>Masa de agua:</b><br>
        <small>{row['masa_agua']}</small><br><br>
        <b>Tendencia:</b> <span style='color:{color};'>{texto_tend}</span><br>
        <b>Cambio:</b> {row['tendencia_m_año']:+.3f} m/año<br>
        <b>R²:</b> {row['r2']:.3f} | <b>p:</b> {row['p_valor']:.3f}<br>
        <b>Nivel medio:</b> {row['nivel_medio']:.1f} m.s.n.m.<br>
        <b>Rango:</b> {row['rango_m']:.1f} m<br>
        <b>Mediciones:</b> {int(row['n_mediciones'])}<br>
        <b>Período:</b> {int(row['año_inicio'])}–{int(row['año_fin'])}
    </div>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color='white',
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['estacion']} | {row['tendencia_m_año']:+.3f} m/año"
    ).add_to(cluster)

# Leyenda
leyenda_html = """
<div style='position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 12px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 12px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.2);'>
    <b style='font-size:13px;'>Tendencia piezométrica</b><br>
    <small>ACA Catalunya — m/año</small><br><br>
    <span style='background:#d62728; border-radius:50%; display:inline-block;
                 width:12px; height:12px;'></span> Descenso fuerte (&lt; −0.5)<br>
    <span style='background:#ff7f0e; border-radius:50%; display:inline-block;
                 width:12px; height:12px;'></span> Descenso moderado (−0.5 a −0.1)<br>
    <span style='background:#bcbd22; border-radius:50%; display:inline-block;
                 width:12px; height:12px;'></span> Estable (−0.1 a +0.1)<br>
    <span style='background:#2ca02c; border-radius:50%; display:inline-block;
                 width:12px; height:12px;'></span> Ascenso moderado (+0.1 a +0.5)<br>
    <span style='background:#1f77b4; border-radius:50%; display:inline-block;
                 width:12px; height:12px;'></span> Ascenso fuerte (&gt; +0.5)<br><br>
    <small>Haz clic en cada piezómetro para ver detalles</small>
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda_html))

# Título
titulo_html = """
<div style='position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
     z-index: 1000; background: rgba(255,255,255,0.92); padding: 8px 16px;
     border-radius: 8px; border: 1px solid #ccc; font-family: Arial;
     font-size: 14px; font-weight: bold; color: #1d3557;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.2);'>
    Red Piezométrica de Catalunya — Tendencias de Niveles Subterráneos | ACA
</div>
"""
mapa.get_root().html.add_child(folium.Element(titulo_html))

folium.LayerControl().add_to(mapa)

# Guardar mapa
mapa.save('output/mapa_piezometrico_catalunya.html')
print('✓ Mapa interactivo guardado: output/mapa_piezometrico_catalunya.html')
print('  Abre este archivo en tu navegador para ver el mapa interactivo.')
mapa

---
## 6. Resumen y conclusiones

In [ ]:
# ── TABLA RESUMEN FINAL ────────────────────────────────────────────────────────
print('=' * 65)
print('   RESUMEN — RED PIEZOMÉTRICA ACA CATALUNYA')
print('=' * 65)

print(f'\n▶ COBERTURA DEL DATASET:')
print(f'   Total registros:        {len(df):,}')
print(f'   Período:                {df["fecha"].min().year} – {df["fecha"].max().year}')
print(f'   Estaciones únicas:      {df["estacion"].nunique():,}')
print(f'   Masas de agua:          {df["masa_agua"].nunique()}')

print(f'\n▶ TENDENCIA GENERAL DE LA RED:')
print(f'   Cambio medio:           {tendencias["tendencia_m_año"].mean():+.3f} m/año')
print(f'   Mediana:                {tendencias["tendencia_m_año"].median():+.3f} m/año')

print(f'\n▶ DISTRIBUCIÓN DE TENDENCIAS:')
cat = [
    ('Descenso fuerte (< −0.5 m/año)',  (tendencias['tendencia_m_año'] < -0.5).sum()),
    ('Descenso moderado (−0.5 a −0.1)', ((tendencias['tendencia_m_año'] >= -0.5) & (tendencias['tendencia_m_año'] < -0.1)).sum()),
    ('Estable (−0.1 a +0.1)',           ((tendencias['tendencia_m_año'] >= -0.1) & (tendencias['tendencia_m_año'] <= 0.1)).sum()),
    ('Ascenso moderado (+0.1 a +0.5)',  ((tendencias['tendencia_m_año'] > 0.1) & (tendencias['tendencia_m_año'] <= 0.5)).sum()),
    ('Ascenso fuerte (> +0.5 m/año)',   (tendencias['tendencia_m_año'] > 0.5).sum()),
]
for label, n in cat:
    pct = n / len(tendencias) * 100
    print(f'   {label}: {n} estaciones ({pct:.0f}%)')

print(f'\n▶ ARCHIVOS GENERADOS:')
archivos = [
    ('output/01_exploratorio.png',              'Análisis exploratorio del dataset'),
    ('output/02_tendencia_general.png',          'Evolución temporal de la red'),
    ('output/03_distribucion_tendencias.png',    'Distribución de tendencias por estación'),
    ('output/04_series_temporales.png',          'Series temporales de principales estaciones'),
    ('output/05_estacionalidad.png',             'Ciclo estacional anual'),
    ('output/mapa_piezometrico_catalunya.html',  'Mapa interactivo Folium'),
]
for archivo, desc in archivos:
    print(f'   ✓ {archivo}')
    print(f'     → {desc}')

print('\n' + '=' * 65)

---
## 7. Discusión

### Interpretación de tendencias
Las tendencias piezométricas reflejan el balance entre recarga natural (precipitación e infiltración) y las extracciones antrópicas. En el contexto mediterráneo de Catalunya, los acuíferos son especialmente vulnerables por:

- **Irregularidad pluviométrica:** precipitaciones concentradas en otoño y primavera, veranos secos.
- **Alta demanda estival:** turismo y riego agrícola coinciden con el período de menor recarga.
- **Cambio climático:** tendencia a veranos más cálidos y secos, reduciendo la recarga.

### Variabilidad estacional
El ciclo anual típico muestra niveles más altos en primavera (tras las lluvias otoñales e invernales y la recarga de nieve en zonas de montaña) y mínimos en verano-otoño (tras la extracción estival y antes de las lluvias de otoño).

### Limitaciones
- La tendencia lineal asume un cambio uniforme en el tiempo, sin considerar cambios en los patrones de extracción o regímenes de precipitación.
- Las estaciones con pocas mediciones (<10) fueron excluidas del análisis de tendencias.
- La conversión de coordenadas puede introducir pequeños errores en la posición de las estaciones si no se usa pyproj.

---

## Referencias

- ACA — Agència Catalana de l'Aigua (2024). *Nivell piezomètric de les aigües subterrànies de Catalunya*. Transparència Catalunya. https://analisi.transparenciacatalunya.cat/d/6899-hrme
- Generalitat de Catalunya (2024). *Pla de gestió del districte de conca fluvial de Catalunya (DCFC) 2022-2027*. ACA.
- Cooper, H.H. & Jacob, C.E. (1946). A generalized graphical method for evaluating formation constants. *Trans. AGU*, 27(4).
